# BattleActs Actuarial Exam Preparation Products
### DATASCI 530 — Project 1
Maxine Hakimi-Bulson

This project collects publicly available information about six BattleActs actuarial exam preparation products: Exams 5, 6C, 6U, 7, 8, and 9. The goal is to construct datasets describing product prices, subscription durations, advertised features, and public sample-resource links.

Data were collected on October 19, 2026. BattleActs granted written permission for this one-time educational scraping project and for sharing the code, dataset, and analysis on GitHub.

Product-page collection recorded sample-link metadata. A subsequent collection downloaded selected public manual PDFs and Excel samples for content extraction; videos and paid materials were excluded.

## Data Collection Process

Before coding, I manually inspected two products and organized their information in a spreadsheet to determine the variables to collect.

The algorithm:
- Opens the BattleActs homepage and the Exams menu.
- Identifies the six exam-product links.
- Uses a loop to click each product link and extract its information.
- Collects sample-card descriptions, button labels, and destination URLs.
- Converts the extracted records into two pandas DataFrames.
- Creates study-tool categories and preliminary preparation-theme indicators from the collected text.
- Uses the saved URLs to download selected public manual PDFs and Excel samples, downloading each unique URL once.
- Extracts PDF text and spreadsheet cell values, records document characteristics, and retains keyword evidence for review.

The final products dataset contains one row per exam, combining product information, study-tool categories, and preparation-theme evidence. The final samples dataset contains one row per advertised card, including extracted document text and measurements for selected files. A third dataset stores sample keyword evidence. Products and samples are linked through the product URL; sample evidence is linked through its destination URL.

Sample resource types are initially inferred from URL extensions. Download and extraction outcomes are recorded for selected PDFs and Excel files; other links remain unverified. Keyword indicators describe wording found in the collected text and require review before interpretation. Sample excerpts are not assumed to represent the entire paid course.

In [ ]:
%pip install requests pypdf openpyxl

In [3]:
# Import packages
import pandas as pd
import time
from selenium import webdriver
import re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Open Chrome and navigate to BattleActs
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

starting_url = "https://battleactsmain.ca/v2/home"
driver.get(starting_url)

## Product Extraction Function

In [4]:
def extract_product(driver):
    # Return None when an optional text field is absent
    def get_text(xpath):
        elements = driver.find_elements("xpath", xpath)
        return elements[0].text.strip() if elements else None

    # Basic product information
    product_name = get_text("//h1")
    price_text = get_text('//h2[contains(., "$")]')

    price = (
        float(price_text.split()[0].replace("$", "").replace(",", ""))
        if price_text else None
    )
    currency = price_text.split()[-1] if price_text else None

    # Description and its bullet points
    description_parts = driver.find_elements(
        "xpath",
        '//h2[contains(., "Study Guide")]/parent::div/p'
        ' | //h2[contains(., "Study Guide")]/parent::div//li'
    )
    description = "\n".join(
        element.text.strip() for element in description_parts
    ) or None

    # Overview feature bullets
    feature_items = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Overview Of Our Features"]'
        '/parent::div/parent::div//ul/li'
    )
    features = "\n".join(
        item.text.strip() for item in feature_items
    ) or None

    # Structured fields derived from the collected text
    feature_text = features or ""
    duration_match = re.search(
        r"(\d+)-month'?s?\s+access", feature_text, re.I
    )
    extension_match = re.search(
        r"free\s+(\d+)-month\s+extension", feature_text, re.I
    )
    practice_match = re.search(
        r"(\d+)\s+original practice exams?", feature_text, re.I
    )

    # Comparison table: retain only feature names and BA descriptions
    study_details = []
    study_rows = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="BattleActs Unique Study Features"]'
        '/following::table[1]/tbody/tr'
    )

    for row in study_rows:
        cells = row.find_elements("xpath", "./td")
        if len(cells) >= 2:
            study_details.append(
                cells[0].text.strip() + ": " + cells[1].text.strip()
            )

    # Introductory text and exam-specific FAQ
    intro_text = get_text(
        '//h2[normalize-space()="Pass your upcoming exam with confidence."]'
        '/following-sibling::p[1]'
    )
    exam_differences = get_text(
        '//p[contains(., "different than other")]'
        '/following-sibling::p[1]'
    )

    product_url = driver.current_url
    collection_date = pd.Timestamp.now(
        tz="America/New_York"
    ).isoformat(timespec="seconds")

    product = {
        "product_name": product_name,
        "price": price,
        "currency": currency,
        "description": description,
        "features": features,
        "duration_months": (
            int(duration_match.group(1)) if duration_match else None
        ),
        "product_url": product_url,
        "collection_date": collection_date,
        "study_features": "\n".join(study_details) or None,
        "exam_differences": exam_differences,
        "intro_text": intro_text,
        "practice_exam_count": (
            int(practice_match.group(1)) if practice_match else None
        ),
        "free_extension_months": (
            int(extension_match.group(1)) if extension_match else None
        )
    }

    # Sample cards within the section containing the sample heading
    sample_cards = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Sample our Study Platform"]'
        '/ancestor::div[contains(concat(" ", normalize-space(@class), " "),'
        ' " product-section ")][1]'
        '//div[contains(concat(" ", normalize-space(@class), " "),'
        ' " owl-item ")'
        ' and not(contains(concat(" ", normalize-space(@class), " "),'
        ' " cloned "))]'
    )

    samples = []

    for card in sample_cards:
        labels = card.find_elements("xpath", ".//p")
        controls = card.find_elements(
            "xpath", ".//a[@href] | .//button[@data-url]"
        )

        if not labels or not controls:
            raise ValueError(
                "Unexpected sample-card structure on " + product_url
            )

        control = controls[0]
        sample_url = (
            control.get_attribute("href")
            or control.get_attribute("data-url")
        )
        url_path = (sample_url or "").split("?")[0].split("#")[0].lower()

        if url_path.endswith(".pdf"):
            resource_type = "PDF"
        elif url_path.endswith((".xlsx", ".xls")):
            resource_type = "Excel"
        elif url_path.endswith(".mp4"):
            resource_type = "Video"
        else:
            resource_type = "Unknown"

        samples.append({
            "product_name": product_name,
            "sample_label": labels[0].get_attribute("textContent").strip(),
            "button_text": control.get_attribute("textContent").strip(),
            "sample_url": sample_url,
            "resource_type": resource_type,
            "product_url": product_url,
            "collection_date": collection_date
        })

    return product, samples

## Automated Navigation and Collection

In [ ]:
wait = WebDriverWait(driver, 15)

# Open the homepage and collect the exam-link destinations
driver.get(starting_url)

menu_xpath = '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
links_xpath = (
    '//a[contains(@class, "dropdown-item")'
    ' and contains(@href, "/v2/product/")]'
)

wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

exam_links = driver.find_elements("xpath", links_xpath)

# Remove repeated URLs while preserving their order
exam_urls = list(dict.fromkeys(
    link.get_attribute("href") for link in exam_links
))

if len(exam_urls) != 6:
    raise ValueError(
        f"Expected 6 exam links, but found {len(exam_urls)}. Check the menu."
    )

products = []
all_samples = []

for exam_url in exam_urls:
    # Reopen the homepage and menu for each product
    driver.get(starting_url)
    wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

    # Navigate by actually clicking the product link
    product_link_xpath = (
        '//a[contains(@class, "dropdown-item")'
        f' and @href="{exam_url}"]'
    )
    wait.until(
        EC.element_to_be_clickable((By.XPATH, product_link_xpath))
    ).click()

    wait.until(EC.url_to_be(exam_url))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h1")))
    time.sleep(2)  # Allow the sample carousel to initialize

    product, samples = extract_product(driver)

    # Preserve completed rows as the loop progresses
    products.append(product)
    all_samples.extend(samples)

    print(
        f'{product["product_name"]}: '
        f'{product["price"]} {product["currency"]}; '
        f'{len(samples)} sample cards'
    )

    time.sleep(3)  # Pause between products

# Build the two datasets
products_df = pd.DataFrame(products)
samples_df = pd.DataFrame(all_samples)

print("Total sample cards:", len(samples_df))

## Sample Availability

In [ ]:
# Classify whether each sample card has a destination
has_url = samples_df["sample_url"].fillna("").str.strip().ne("")
coming_soon = samples_df["button_text"].str.contains(
    "coming soon", case=False, na=False
)

samples_df["sample_status"] = "No URL listed"
samples_df.loc[has_url, "sample_status"] = "Link listed"
samples_df.loc[~has_url & coming_soon, "sample_status"] = "Coming soon"

# Summarize sample cards by product and status
sample_summary = pd.crosstab(
    samples_df["product_name"],
    samples_df["sample_status"]
)



## Additional Data Preparation

Study tools are classified using product descriptions, feature lists,
comparison-table text, and sample labels. Public sample formats are
recorded separately from advertised product features.

Preparation themes are identified using keyword rules applied to the
introductory text and exam-specific FAQ. These are preliminary indicators
of the provider's wording, not measures of exam difficulty. Matching
sentences are retained for review.

In [ ]:
import requests
from pathlib import Path
from urllib.parse import urlsplit, unquote
from pypdf import PdfReader
from openpyxl import load_workbook

# Work on a separate table to preserve the original product dataset
product_categories_df = products_df[[
    "product_name", "product_url"
]].copy()

# Combine relevant product sections and sample labels
sample_labels = samples_df.groupby("product_url")["sample_label"].agg(
    lambda labels: "\n".join(labels)
)

tool_text = (
    products_df[["description", "features", "study_features"]]
    .fillna("")
    .agg("\n".join, axis=1)
    + "\n"
    + products_df["product_url"].map(sample_labels).fillna("")
)

# Indicators describe mentions in the collected text, not verified availability
tool_patterns = {
    "manual_mentioned": r"\bmanual\b",
    "instructional_video_mentioned": r"video lessons|instructional videos?",
    "formula_summary_sheet_mentioned": r"formula|summary sheets?",
    "quizzes_flashcards_mentioned": r"quizzes|flashcards",
    "excel_practice_mentioned": r"\bExcel\b",
    "discussion_support_mentioned": r"discussion forum|instructor support",
    "study_schedule_mentioned": r"study schedule",
    "hybrid_exam_mentioned": r"hybrid CAS exam"
}

for column, pattern in tool_patterns.items():
    product_categories_df[column] = tool_text.str.contains(
        pattern, case=False, regex=True, na=False
    )

# Record public sample formats separately
for resource in ["PDF", "Excel", "Video"]:
    counts = (
        samples_df.loc[samples_df["resource_type"].eq(resource)]
        .groupby("product_url")
        .size()
    )
    product_categories_df[f"{resource.lower()}_sample_cards"] = (
        products_df["product_url"].map(counts).fillna(0).astype(int)
    )


# Use exam-specific text rather than general marketing features for themes
theme_patterns = {
    "memorization": r"memoriz\w*",
    "calculation": r"calculat\w*",
    "conceptual_understanding": r"concepts?|understanding|mastery",
    "time_pressure": r"time pressure|time management|rapidly|quickly|limited time",
    "integration": r"integrative|integrat\w*|synthesi\w*|connect"
}

theme_records = []

for _, product in products_df.iterrows():
    text = "\n".join(
        str(product[column]) if pd.notna(product[column]) else ""
        for column in ["intro_text", "exam_differences"]
    )
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text)

    for theme, pattern in theme_patterns.items():
        evidence = [
            sentence.strip()
            for sentence in sentences
            if re.search(pattern, sentence, re.I)
        ]

        theme_records.append({
            "product_name": product["product_name"],
            "product_url": product["product_url"],
            "theme": theme,
            "keyword_match": bool(evidence),
            "evidence": " ".join(evidence)
        })

preparation_themes_df = pd.DataFrame(theme_records)


# Review evidence before treating matches as final classifications


## Public Sample Content Collection

The additional collection includes cards labeled as study-manual samples
and cards with Excel destination URLs. Video files are excluded. Each
unique URL is downloaded once, with a pause between requests.

PDF text and nonempty spreadsheet cell values are extracted. Spreadsheet
formulas are read as text and are not executed. These excerpts may cover
different topics and are not assumed to represent the entire paid course.

In [12]:
# Select manual PDFs and linked Excel samples
manual_pdf = (
    samples_df["resource_type"].eq("PDF")
    & samples_df["sample_label"].str.contains(
        "study manual", case=False, na=False
    )
)

excel_sample = samples_df["resource_type"].eq("Excel")
has_url = samples_df["sample_url"].fillna("").str.strip().ne("")

selected_samples = samples_df.loc[
    (manual_pdf | excel_sample) & has_url
].copy()

sample_folder = Path("sample_files")
sample_folder.mkdir(exist_ok=True)

download_records = []

for number, url in enumerate(
    selected_samples["sample_url"].drop_duplicates(), start=1
):
    suffix = Path(unquote(urlsplit(url).path)).suffix.lower()
    local_path = sample_folder / f"sample_{number:02d}{suffix}"

    record = {
        "sample_url": url,
        "local_file": str(local_path),
        "content_collection_date": pd.Timestamp.now(
            tz="America/New_York"
        ).isoformat(timespec="seconds"),
        "extraction_status": "Failed",
        "extraction_error": None,
        "page_count": None,
        "sheet_count": None,
        "sheet_names": None,
        "nonempty_cells": None,
        "formula_cells": None,
        "extracted_text": ""
    }

    try:
        # Reuse an existing download if this cell is run again
        if not local_path.exists():
            response = requests.get(
                url,
                headers={"User-Agent": "DATASCI530 educational project"},
                timeout=60
            )
            response.raise_for_status()
            local_path.write_bytes(response.content)

        if suffix == ".pdf":
            reader = PdfReader(local_path)
            record["page_count"] = len(reader.pages)
            text = "\n".join(
                page.extract_text() or "" for page in reader.pages
            )

        elif suffix == ".xlsx":
            workbook = load_workbook(
                local_path, read_only=True, data_only=False
            )

            try:
                record["sheet_count"] = len(workbook.sheetnames)
                record["sheet_names"] = " | ".join(workbook.sheetnames)

                text_parts = []
                cell_count = 0
                formula_count = 0

                for sheet in workbook.worksheets:
                    text_parts.append(f"Worksheet: {sheet.title}")

                    for row in sheet.iter_rows():
                        values = []

                        for cell in row:
                            if cell.value is None:
                                continue

                            cell_count += 1
                            formula_count += int(cell.data_type == "f")
                            values.append(str(cell.value))

                        if values:
                            text_parts.append(" | ".join(values))

                record["nonempty_cells"] = cell_count
                record["formula_cells"] = formula_count
                text = "\n".join(text_parts)

            finally:
                workbook.close()

        else:
            raise ValueError(f"Unsupported sample format: {suffix}")

        record["extracted_text"] = text.strip()
        record["extraction_status"] = (
            "Extracted" if text.strip() else "No text extracted"
        )

    except Exception as error:
        # Keep failed samples in the dataset for quality checking
        record["extraction_error"] = (
            f"{type(error).__name__}: {error}"
        )

    download_records.append(record)
    print(local_path.name, record["extraction_status"])
    time.sleep(3)

downloaded_samples_df = pd.DataFrame(download_records)

sample_content_df = selected_samples.merge(
    downloaded_samples_df,
    on="sample_url",
    how="left",
    validate="many_to_one"
)

sample_01.pdf Extracted
sample_02.xlsx Extracted
sample_03.pdf Extracted
sample_04.xlsx Extracted
sample_05.pdf Extracted
sample_06.xlsx Extracted
sample_07.xlsx Extracted
sample_08.pdf Extracted
sample_09.xlsx Extracted
sample_10.pdf Extracted
sample_11.xlsx Extracted
sample_12.pdf Extracted


## Sample Content Indicators and Quality Checks

In [ ]:
content_patterns = {
    "worked_example": r"\bworked examples?\b|\bstep.by.step\b",
    "practice_question": r"\bquestions?\b|\bproblems?\b|\bexercises?\b",
    "solution": r"\bsolutions?\b|\banswers?\b"
}

content_evidence = []

for indicator, pattern in content_patterns.items():
    column = f"{indicator}_keyword"

    # Missing means extraction failed or yielded no text
    sample_content_df[column] = pd.Series(
        pd.NA, index=sample_content_df.index, dtype="boolean"
    )

    extracted = sample_content_df["extraction_status"].eq("Extracted")

    sample_content_df.loc[extracted, column] = (
        sample_content_df.loc[extracted, "extracted_text"]
        .str.contains(pattern, case=False, regex=True)
    )

    for _, sample in sample_content_df.loc[extracted].iterrows():
        match = re.search(pattern, sample["extracted_text"], re.I)

        if match:
            text = sample["extracted_text"]
            content_evidence.append({
                "product_name": sample["product_name"],
                "sample_url": sample["sample_url"],
                "indicator": indicator,
                "evidence": text[
                    max(0, match.start() - 100):match.end() + 150
                ].replace("\n", " ")
            })

sample_evidence_df = pd.DataFrame(
    content_evidence,
    columns=["product_name", "sample_url", "indicator", "evidence"]
)



## Consolidated Datasets

Three datasets are exported: Products (one row per exam), Samples (one row per advertised card, including extracted text where selected), and Sample Evidence (one row per sample–keyword match). Preparation-theme sentences are stored in Products. Unselected cards remain in Samples with an explicit extraction status.

In [ ]:
# Combine product categories and theme evidence without repeating product rows
category_fields = product_categories_df.drop(columns="product_name")
products_final = products_df.merge(
    category_fields, on="product_url", how="left", validate="one_to_one"
)

for theme in preparation_themes_df["theme"].unique():
    theme_rows = preparation_themes_df.loc[
        preparation_themes_df["theme"].eq(theme)
    ].set_index("product_url")
    products_final[f"{theme}_mentioned"] = products_final["product_url"].map(
        theme_rows["keyword_match"]
    )
    products_final[f"{theme}_evidence"] = products_final["product_url"].map(
        theme_rows["evidence"]
    ).fillna("")

# Keep all 25 advertised cards, attaching content only to selected samples
sample_keys = ["product_url", "sample_label", "sample_url"]
content_fields = sample_keys + [
    column for column in sample_content_df.columns
    if column not in samples_df.columns
]
samples_final = samples_df.merge(
    sample_content_df[content_fields],
    on=sample_keys, how="left", validate="one_to_one"
)
not_selected = samples_final["extraction_status"].isna()
samples_final.loc[not_selected, "extraction_status"] = "Not selected"
samples_final.loc[
    not_selected & samples_final["sample_status"].eq("Coming soon"),
    "extraction_status"
] = "No URL listed"

# Verify the merges preserved observations, text, and relationships
assert len(products_final) == len(products_df)
assert len(samples_final) == len(samples_df)
assert products_final["product_url"].is_unique
assert not samples_final.duplicated(sample_keys).any()
assert samples_final["product_url"].isin(products_final["product_url"]).all()
assert sample_evidence_df["sample_url"].isin(samples_final["sample_url"]).all()
assert (
    samples_final["extraction_status"].eq("Extracted").sum()
    == sample_content_df["extraction_status"].eq("Extracted").sum()
)
assert "formula_keyword" not in samples_final.columns


## Results

### Table 1. Dataset Completeness

Missing/blank cells include intentional blanks: unmatched theme evidence, inapplicable PDF/Excel measurements, empty error fields after successful extraction, and content fields for unselected cards. They are not all collection failures.

In [ ]:
# Missing cells include intentional blanks, reported separately below
frames = {"Products": products_final, "Samples": samples_final,
          "Sample evidence": sample_evidence_df}
quality_table = pd.DataFrame([
    {"Dataset": name, "Observations": len(frame),
     "Missing/blank cells": int(frame.replace(r"^\s*$", pd.NA, regex=True)
                               .isna().sum().sum())}
    for name, frame in frames.items()
])
display(quality_table)
print("Duplicate product URLs:", products_final["product_url"].duplicated().sum())
print("Duplicate sample rows:", samples_final.duplicated().sum())
print("Sample cards missing a URL:", samples_final["sample_url"].isna().sum())
print("Content extraction outcomes:",
      samples_final["extraction_status"].value_counts().to_dict())


,Dataset,Observations,Missing/blank cells
0,Products,6,8
1,Samples,25,199
2,Sample evidence,24,0


## Data Quality

The collection produced six product observations and 25 sample-card observations. No product URLs or complete sample rows were duplicated. All product fields were populated.

One sample URL was missing. Inspection showed that the associated Exam 9 card was labeled "Coming Soon"; this record was retained and classified accordingly. Shared destination URLs were also retained because separate sample cards can advertise different resources using the same link.

Exam 5 results were compared with the manually collected values: USD 500, seven months of access, a conditional six-month extension, and four original practice exams.

Prices were stored numerically, with currency in a separate column. USD and CAD prices were not pooled when calculating price statistics.

Text extraction succeeded for all 12 selected samples: six manual PDFs and six Excel workbooks. PDF samples contain 5–17 pages, while Excel samples contain 3–32 worksheets. Spreadsheet counts include instruction and contents sheets, so they do not represent numbers of practice questions. Missing format-specific measurements are not applicable: PDFs have no worksheet counts, and workbooks have no PDF page counts.

Keyword indicators record matching words, not verified learning activities. Actual Excel formula cells are counted separately from keyword indicators.

### Table 2. Numeric Summaries

Variances are sample variances. Price variances use squared currency units. The CAD variance is undefined with one observation.

In [ ]:
# Prices are kept separate by currency; other measures use all products
summary_records = []
for currency, group in products_final.groupby("currency"):
    values = group["price"]
    summary_records.append({"Measure": f"Price ({currency})",
                            "Observations": values.count(),
                            "Mean": values.mean(), "Sample variance": values.var()})
for column, label in {
    "duration_months": "Standard access (months)",
    "practice_exam_count": "Original practice exams",
    "free_extension_months": "Conditional extension (months)"
}.items():
    values = products_final[column]
    summary_records.append({"Measure": label, "Observations": values.count(),
                            "Mean": values.mean(), "Sample variance": values.var()})
display(pd.DataFrame(summary_records).round(2))


,Measure,Observations,Mean,Sample variance
0,Price (CAD),1,1000.00,NaN
1,Price (USD),5,900.00,50000.00
2,Standard access (months),6,7.00,0.00
3,Original practice exams,6,2.17,0.97
4,Conditional extension (months),6,6.00,0.00


### Table 3. Study Tools Mentioned by Product

“Not matched” means the rule found no match in the collected sections; it does not prove a feature is unavailable. Video sample-card counts are stored separately because promotional videos are not necessarily instructional lessons.

In [ ]:
tool_columns = {
    "manual_mentioned": "Manual",
    "instructional_video_mentioned": "Video lessons",
    "formula_summary_sheet_mentioned": "Formula/summary sheets",
    "quizzes_flashcards_mentioned": "Quizzes/flashcards",
    "excel_practice_mentioned": "Excel",
    "discussion_support_mentioned": "Forum/support",
    "study_schedule_mentioned": "Study schedule",
    "hybrid_exam_mentioned": "Hybrid exam"
}
tool_table = products_final.set_index("product_name")[list(tool_columns)]
tool_table = tool_table.rename(columns=tool_columns).replace(
    {True: "Mentioned", False: "Not matched"}
)
tool_table.index.name = "Product"
display(tool_table)


,Manual,Video lessons,Formula/summary sheets,Quizzes/flashcards,Excel,Forum/support,Study schedule,Hybrid exam
Product,,,,,,,,
CAS Exam 5,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Not matched
CAS Exam 6C,Mentioned,Not matched,Not matched,Mentioned,Mentioned,Mentioned,Mentioned,Not matched
CAS Exam 6U,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Not matched
CAS Exam 7,Mentioned,Not matched,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned
CAS Exam 8,Mentioned,Not matched,Not matched,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned
CAS Exam 9,Mentioned,Not matched,Not matched,Mentioned,Mentioned,Mentioned,Mentioned,Mentioned


### Table 4. Selected Sample Characteristics

Each row is a selected sample card. Worksheets include instruction and contents sheets. Missing PDF/Excel measurements are not applicable. Page and cell counts do not measure course quality.

In [ ]:
selected = samples_final.loc[
    samples_final["extraction_status"].ne("Not selected")
    & samples_final["extraction_status"].ne("No URL listed")
]
sample_overview = selected[[
    "product_name", "resource_type", "extraction_status",
    "page_count", "sheet_count", "nonempty_cells", "formula_cells"
]].rename(columns={
    "product_name": "Product", "resource_type": "Format",
    "extraction_status": "Extraction", "page_count": "PDF pages",
    "sheet_count": "Worksheets", "nonempty_cells": "Nonempty cells",
    "formula_cells": "Formula cells"
})
for column in ["PDF pages", "Worksheets", "Nonempty cells", "Formula cells"]:
    sample_overview[column] = sample_overview[column].astype("Int64")
display(sample_overview)


,Product,Format,Extraction,PDF pages,Worksheets,Nonempty cells,Formula cells
0,CAS Exam 5,PDF,Extracted,17,<NA>,<NA>,<NA>
4,CAS Exam 5,Excel,Extracted,<NA>,32,1832,325
6,CAS Exam 6C,PDF,Extracted,7,<NA>,<NA>,<NA>
8,CAS Exam 6C,Excel,Extracted,<NA>,11,2762,288
9,CAS Exam 6U,PDF,Extracted,5,<NA>,<NA>,<NA>
11,CAS Exam 6U,Excel,Extracted,<NA>,29,1197,275
14,CAS Exam 6U,Excel,Extracted,<NA>,3,476,70
15,CAS Exam 7,PDF,Extracted,9,<NA>,<NA>,<NA>
18,CAS Exam 7,Excel,Extracted,<NA>,4,283,42
19,CAS Exam 8,PDF,Extracted,6,<NA>,<NA>,<NA>


## Discussion

The five USD-priced products have an average listed price of $900. Exam 6C is priced separately at CAD 1,000. All six products advertise seven months of standard access and a conditional six-month free extension.

The public sample section contains 25 advertised cards, of which 24 have destination URLs. One Exam 9 card is marked "Coming Soon." Some cards share a destination, so card counts do not represent counts of unique files.

These data describe one provider's public offerings at one point in time. They do not measure course quality, exam outcomes, or independently establish differences between BattleActs and its competitors.

The selected public sample collection contains six manual PDFs and six Excel workbooks, all of which yielded text. Keyword matches provide preliminary evidence for later thematic analysis; they do not establish the absence of a tool or learning activity. Differences in sample length or workbook structure do not measure course quality.

## Export

These three CSV files are the final datasets. Intermediate DataFrames remain in memory for reproducibility and are not separately exported.

In [ ]:
products_final.to_csv("battleacts_products.csv", index=False)
samples_final.to_csv("battleacts_samples.csv", index=False)
sample_evidence_df.to_csv("battleacts_sample_evidence.csv", index=False)


In [15]:
driver.quit()